In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#special case company
"""
"4348"
'2285', '4081', '2286', '4340', '4342', '2283', 
'2284', '1323', '4009', '4334', '4084', '4261', 
'4012', '1830', '4333', '1835', '4165', '4347', 
'2382', '2223', '4163', '4262', '4083', '4164', 
'1834', '1303', '2081', '4162', '7204', '7202', 
'2281', '1833', '4143', '4142', '4263', '1831', 
'4051', '1322', '1182', '6017', '4161', '4031', 
'4071', '4322', '4264', '4016', '4072', '4191', 
'4332', '2381', '6014', '4017', '4015', '4324', 
'6015', '4336', '4338', '4082', '4144', '4014', 
'1111', '4291', '4339', '4013', '2282', '4325', 
'1304', '7203', '4018', '2084', '7200', '2287', 
'4331', '8313', '4193', '1183', '4192', '4345', 
'4344', '6016', '2082', '2083', '6013', '3007',

# failed training
"4003","1010", "3030","8200","1150", "2020", "1140","7020",
    "8100", "2060", "1180", "3080", 
    "2150", "4260",
    "3020", "1320", "4007", "3040",
    "8280", "6070",  "2270", "3003",  

"""


"""

# done training (4 Output already)
"3090","8300","1050"
"2050", "4001",

"1030","8170","3030","1020","3010","2060", "1180"

"4250","2280","4020","3080"

"8160"

# on hold 
"1214", "8010", "3002", "3050", "8030",  "3005",
    "2230", "4280",  "4004", "8210",  "1212",
        "1010", "8100", "1120","4002", "8012",
        "4150", "8070","3004", "2080",
    "3060", "1302", "4300", "4090", "6001", "6004",
    "7040", "2310", "2300", "8230", "4200", "4100",
"2190", "2250", "1211", "8180", "2290", "2240", "4180",
"8240", "4210", "8040", "8310", "4050", "4040", "1210", 
"""


"""" 
    
    "4006","2070","6020","1301","4130","4290"

"""

# no 5D

""""

"8020","4006","2070","6020","1301","4130",
"""
# 5D
""""
    "1030", "1180","8300","8170","3030","3090","1050","4290"

    #hard watch for result
    "4001","1020","2050","3010"
"""




symbols = [
    "2060","4250","2280","4020","3080","8160"
    ]



dfs = {}

for s in symbols:
    filepath = f"{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── 2060 ──────────────────────────
  Shape : (5558, 46)
  Head  :
              datetime  ticker  open_price  high_price  low_price  \
0  2004-02-24 16:00:00    2060   12.267563   12.654960  12.224519   
1  2004-02-25 16:00:00    2060   12.554524   13.831498  12.511480   
2  2004-02-28 16:00:00    2060   13.415405   14.018023  13.200185   
3  2004-02-29 16:00:00    2060   13.573234   13.673670  13.314969   
4  2004-03-01 16:00:00    2060   13.343665   13.343665  13.085401   

   close_price        volume     EMA_10     EMA_20  EMA_ratio  ...  \
0    12.583220  1.921707e+07  12.224360  11.929544   1.054795  ...   
1    13.601930  3.922330e+07  12.474827  12.088819   1.125166  ...   
2    13.573234  3.085537e+07  12.674538  12.230192   1.109814  ...   
3    13.358013  5.573362e+06  12.798806  12.337603   1.082707  ...   
4    13.228881  5.270603e+06  12.877001  12.422487   1.064914  ...   

   volume_lag_3  volume_lag_4  volume_lag_5  above_ema10  above_ema20  \
0  2.514527e+06  2.34575

In [3]:
for symbol in symbols:
    df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)
    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.dropna().reset_index(drop=True)

    for col, sigma in {'price_pct_change': 3, 'volume_pct_change_log': 2}.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # Date-based split: train = up to 2018-12-31, test = 2019 onwards
    train_df = df[df['datetime'] <= '2018-12-31']
    test_df  = df[df['datetime'] >  '2018-12-31']

    target_scaler = StandardScaler()
    train_y = target_scaler.fit_transform(train_df[['price_pct_change', 'volume_pct_change_log']])

    print(f"\n{symbol}")
    print(f"  Raw train UP%      : {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scaled train mean  : {train_y[:, 0].mean():.4f}")
    print(f"  Scaled train std   : {train_y[:, 0].std():.4f}")
    print(f"  Train date range   : {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Test  date range   : {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")
    print(f"  Train rows: {len(train_df)}  Test rows: {len(test_df)}")


2060
  Raw train UP%      : 46.57%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2004-03-03 → 2018-12-30
  Test  date range   : 2018-12-31 → 2026-06-08
  Train rows: 3367  Test rows: 1807

4250
  Raw train UP%      : 42.26%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2008-01-29 → 2018-12-30
  Test  date range   : 2018-12-31 → 2026-06-08
  Train rows: 2508  Test rows: 1782

2280
  Raw train UP%      : 46.58%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2005-10-16 → 2018-12-27
  Test  date range   : 2019-01-01 → 2026-06-08
  Train rows: 3027  Test rows: 1772

4020
  Raw train UP%      : 45.67%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Train date range   : 2001-08-13 → 2018-12-30
  Test  date range   : 2018-12-31 → 2026-06-08
  Train rows: 4018  Test rows: 1832

3080
  Raw train UP%      : 44.54%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.000

In [4]:
import pickle
import os

SEQ_LEN_DEFAULT = 30
prepared = {}
scalers  = {}

FEATURE_COLS = [
    "open_price", "high_price", "low_price",
    "close_price", "volume",
    "EMA_10", "EMA_20", "EMA_ratio", "MACD_hist",
    "RSI_14", "ROC_5", "ROC_10", "ROC_20",
    "ATR_14", "ATR_ratio", "BB_pct", "realized_vol",
    "OBV", "OBV_momentum", "volume_ratio",
    "volume_surge", "MFI_14",
    "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_4", "close_lag_5",
    "returns_lag_1", "returns_lag_2", "returns_lag_3", "returns_lag_4", "returns_lag_5",
    "volume_lag_1", "volume_lag_2", "volume_lag_3", "volume_lag_4", "volume_lag_5",
    "above_ema10", "above_ema20", "roc5_pos", "roc20_pos", "up_streak",
]

TARGET_COLS = ["price_pct_change_5d"]


def _make_seqs(X, y, seq_len):
    """Simple sliding-window builder."""
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        ws.append(1.0)
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


for symbol in symbols:
    filepath = f"{symbol}_daily.csv"
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue

    df = pd.read_csv(filepath, parse_dates=["datetime"])

    # ── Lag features ──────────────────────────────────────────────────────────
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # ── Regime / persistence features ─────────────────────────────────────────
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)

    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up

    # ── 5-day forward close % change target ───────────────────────────────────
    df["price_pct_change_5d"] = (df["close_price"].shift(-5) / df["close_price"] - 1) * 100

    df = df.dropna().reset_index(drop=True)

    # ── Clip outliers ─────────────────────────────────────────────────────────
    for col, sigma in {"price_pct_change_5d": 3}.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)


    train_df = df[df['datetime'] <= '2018-12-31'].iloc[:-5]   # 5d target of last rows peeks into 2019 test prices
    test_df  = df[df['datetime'] >  '2018-12-31']

    train_X_raw = train_df[FEATURE_COLS].values.astype(np.float32)
    test_X_raw  = test_df[FEATURE_COLS].values.astype(np.float32)

    train_y_raw = train_df[TARGET_COLS].values.astype(np.float32)
    test_y_raw  = test_df[TARGET_COLS].values.astype(np.float32)

    # ── Fit scalers on train only ─────────────────────────────────────────────
    feature_scaler = StandardScaler()
    target_scaler  = StandardScaler()

    train_X_scaled = feature_scaler.fit_transform(train_X_raw)
    test_X_scaled  = feature_scaler.transform(test_X_raw)

    train_y_scaled = target_scaler.fit_transform(train_y_raw)
    test_y_scaled  = target_scaler.transform(test_y_raw)

    # ── Save scalers ──────────────────────────────────────────────────────────
    os.makedirs(f"models/{symbol}", exist_ok=True)
    with open(f"models/{symbol}/{symbol}_feature_scaler.pkl", "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "wb") as f:
        pickle.dump(target_scaler, f)

    # ── Pre-build sequences at default SEQ_LEN ────────────────────────────────
    X_train, y_train, _ = _make_seqs(train_X_scaled, train_y_scaled, SEQ_LEN_DEFAULT)
    X_test,  y_test,  _ = _make_seqs(test_X_scaled,  test_y_scaled,  SEQ_LEN_DEFAULT)

    prepared[symbol] = {
        "train_X_scaled": train_X_scaled,
        "train_y_scaled": train_y_scaled,
        "test_X_scaled":  test_X_scaled,
        "test_y_scaled":  test_y_scaled,
        "X_train": X_train,  "y_train": y_train,
        "X_test":  X_test,   "y_test":  y_test,
    }
    scalers[symbol] = {
        "feature_scaler": feature_scaler,
        "target_scaler":  target_scaler,
    }

    print(f"\n── {symbol} {'─' * 60}")
    print(f"  Rows  → train:{len(train_df)}  test:{len(test_df)}")
    print(f"  Seqs  → train:{X_train.shape[0]}  test:{X_test.shape[0]}")
    print(f"  Shape → X:{X_train.shape}  y:{y_train.shape}")
    print(f"  Train date range: {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Test  date range: {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")
    print(f"  Scalers saved → models/{symbol}/")

print(f"\n── prepared built for {len(prepared)} symbols ──")


── 2060 ────────────────────────────────────────────────────────────
  Rows  → train:3597  test:1845
  Seqs  → train:3567  test:1815
  Shape → X:(3567, 30, 42)  y:(3567, 1)
  Train date range: 2004-03-03 → 2018-12-23
  Test  date range: 2018-12-31 → 2026-06-02
  Scalers saved → models/2060/

── 4250 ────────────────────────────────────────────────────────────
  Rows  → train:2677  test:1824
  Seqs  → train:2647  test:1794
  Shape → X:(2647, 30, 42)  y:(2647, 1)
  Train date range: 2008-01-29 → 2018-12-23
  Test  date range: 2018-12-31 → 2026-06-02
  Scalers saved → models/4250/

── 2280 ────────────────────────────────────────────────────────────
  Rows  → train:3189  test:1846
  Seqs  → train:3159  test:1816
  Shape → X:(3159, 30, 42)  y:(3159, 1)
  Train date range: 2005-10-12 → 2018-12-23
  Test  date range: 2018-12-31 → 2026-06-02
  Scalers saved → models/2280/

── 4020 ────────────────────────────────────────────────────────────
  Rows  → train:4141  test:1841
  Seqs  → train:411

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32) \
                 if weights is not None \
                 else torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


BATCH_SIZE = 32
loaders    = {}

for symbol, data in prepared.items():
    train_loader = DataLoader(TadawulDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(TadawulDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Test: {len(test_loader)} batches")

    sample_X, sample_y, sample_w = next(iter(train_loader))
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 1)")
    print(f"  Sample w: {sample_w.shape} → (batch,)")


2060 — Train: 112 batches | Test: 57 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

4250 — Train: 83 batches | Test: 57 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

2280 — Train: 99 batches | Test: 57 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

4020 — Train: 129 batches | Test: 57 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

3080 — Train: 112 batches | Test: 57 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

8160 — Train: 81

In [6]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}

# ── BiLSTM + Attention Model (with LayerNorm) ─────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.attention      = nn.Linear(hidden_size * 2, 1)
        self.pos_bias       = nn.Parameter(torch.zeros(1))
        self.ln             = nn.LayerNorm(hidden_size * 2)
        self.dropout        = nn.Dropout(dropout)
        self.fc             = nn.Linear(hidden_size * 2, 1)
        self.skip_fc        = nn.Linear(hidden_size * 2, 1)
        self.output_blend   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        seq_len      = lstm_out.size(1)
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        out_attn     = self.fc(self.dropout(self.ln(context)))
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


# ── Joint Loss ────────────────────────────────────────────────────────────────
class JointPredictionLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=0.5, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma

    def forward(self, preds, targets, weights=None):
        # Plain SmoothL1 (Huber, delta=1.0) on the standardized target.
        # Anti-collapse stack removed (BCE / std_match / collapse / sign):
        # it drove the mean-drift collapse. alpha/beta/gamma/weights kept in
        # the signature only for call-site compatibility; unused.
        return nn.SmoothL1Loss(beta=1.0)(preds, targets)


# ── Helpers ───────────────────────────────────────────────────────────────────
def create_sequences(X, y, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        move   = abs(y[i + seq_len, 0])
        ws.append(1.0 + 2.0 * float(move > move_threshold))
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


def make_balanced_sampler(y, oversample_factor=2.0):
    labels = (y[:, 0] > 0).astype(int)
    moves  = np.abs(y[:, 0])
    median_move = np.median(moves)
    bucket = labels * 2 + (moves > median_move).astype(int)
    counts = np.bincount(bucket, minlength=4).astype(float)
    counts = np.maximum(counts, 1)
    weight_map = 1.0 / counts
    weight_map[1] *= oversample_factor
    weight_map[3] *= oversample_factor
    sample_weights = torch.tensor(weight_map[bucket], dtype=torch.float32)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


def safe_corr(a, b):
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else float(r)


# ── Clean NaN values ──────────────────────────────────────────────────────────
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    for split in ("train", "test"):
        X_key, y_key = f"{split}_X_scaled", f"{split}_y_scaled"
        X, y         = data[X_key], data[y_key]
        mask         = ~(np.isnan(X).any(axis=1) | np.isnan(y).any(axis=1))
        before       = len(X)
        data[X_key]  = X[mask]
        data[y_key]  = y[mask]
        print(f"  {symbol} {split}: {before} → {len(data[X_key])} rows")


# ── Optuna search ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Optuna search — {symbol}")
    print(f"{'═'*55}")

    train_X    = data["train_X_scaled"]
    train_y    = data["train_y_scaled"]
    val_X      = data["test_X_scaled"]
    val_y      = data["test_y_scaled"]
    input_size = train_X.shape[1]

    def objective(trial):
        hidden_size    = trial.suggest_categorical("hidden_size", [64, 128])
        num_layers     = trial.suggest_int("num_layers", 2, 3)
        dropout        = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
        lr             = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        batch_size     = trial.suggest_categorical("batch_size", [32, 64])
        seq_len        = trial.suggest_categorical("seq_len", [10, 20, 30, 40])
        alpha          = trial.suggest_float("alpha", 3.0, 12.0, step=0.5)
        beta           = trial.suggest_float("beta", 0.3, 1.0, step=0.1)
        move_threshold = trial.suggest_float("move_threshold", 0.5, 2.0, step=0.1)
        gamma          = trial.suggest_float("gamma", 0.5, 3.0, step=0.5)

        X_tr, y_tr, w_tr = create_sequences(train_X, train_y, seq_len, move_threshold)
        X_vl, y_vl, _    = create_sequences(val_X,   val_y,   seq_len, move_threshold)

        train_ld = DataLoader(
            TadawulDataset(X_tr, y_tr, w_tr),
            batch_size=batch_size,
            shuffle=True,
            drop_last=True
        )
        val_ld = DataLoader(
            TadawulDataset(X_vl, y_vl),
            batch_size=batch_size,
            shuffle=False,
            drop_last=False
        )

        SEEDS       = [42, 123, 456]
        seed_scores = []

        for seed_idx, seed in enumerate(SEEDS):
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = StockPctChangeBiLSTMAttention(
                input_size  = input_size,
                hidden_size = hidden_size,
                num_layers  = num_layers,
                dropout     = dropout
            ).to(device)

            criterion = JointPredictionLoss(alpha=alpha, beta=beta, gamma=gamma)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

            EPOCHS         = 50
            PATIENCE       = 10
            best_val_score = 0.0
            patience_ctr   = 0
            seed_collapsed = False

            for epoch in range(1, EPOCHS + 1):
                model.train()
                for X_batch, y_batch, w_batch in train_ld:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    w_batch = w_batch.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(X_batch), y_batch, w_batch)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                model.eval()
                all_preds, all_actuals = [], []
                with torch.no_grad():
                    for X_batch, y_batch, _ in val_ld:
                        all_preds.append(model(X_batch.to(device)).cpu().numpy())
                        all_actuals.append(y_batch.numpy())

                if len(all_preds) == 0:
                    seed_collapsed = True
                    break

                val_preds   = np.vstack(all_preds)
                val_actuals = np.vstack(all_actuals)

                price_dir = balanced_accuracy_score(
                    np.sign(val_actuals[:, 0]).astype(int),
                    np.sign(val_preds[:, 0]).astype(int)
                )

                price_corr  = safe_corr(val_actuals[:, 0], val_preds[:, 0])

                pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]), 1e-6)
                pred_up_pct    = (val_preds[:, 0] > 0).mean()
                actual_up_pct  = (val_actuals[:, 0] > 0).mean()
                bias_gap       = abs(pred_up_pct - actual_up_pct)

                if epoch >= 5:
                    if pred_up_pct < 0.15 or pred_up_pct > 0.85:
                        seed_collapsed = True
                        break

                balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0

                val_score = (
                    0.60 * price_dir +
                    0.40 * max(price_corr, 0.0) -
                    abs(1.0 - pred_std_ratio) * 0.3 -
                    bias_gap * 3.0 -
                    balance_penalty
                )

                if val_score > best_val_score:
                    best_val_score = val_score
                    patience_ctr   = 0
                else:
                    patience_ctr += 1
                    if patience_ctr >= PATIENCE:
                        break

            seed_scores.append(-1.0 if seed_collapsed else best_val_score)

            trial.report(float(np.mean(seed_scores)), seed_idx)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return float(np.mean(seed_scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        study_name=f"bilstm_attention_{symbol}"
    )
    study.optimize(objective, n_trials=50, timeout=3600)

    best_params_all[symbol] = study.best_params

    print(f"\n  Best joint score : {study.best_value:.4f}")
    print(f"  Best trial       : {study.best_trial.number}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k:15s} : {v}")

    print(f"\n  Top 5 Trials:")
    trials_df = study.trials_dataframe().sort_values("value", ascending=False).head()
    cols = [
        "number", "value",
        "params_hidden_size", "params_num_layers", "params_dropout",
        "params_lr", "params_batch_size", "params_seq_len",
        "params_alpha", "params_beta", "params_move_threshold",
    ]
    print(trials_df[[c for c in cols if c in trials_df.columns]])

print(f"\n── Optuna done for {len(best_params_all)} symbols ──")

Using device: cuda

[I 2026-06-15 09:00:38,646] A new study created in memory with name: bilstm_attention_2060




Cleaning NaN values from all symbols...
  2060 train: 3597 → 3597 rows
  2060 test: 1845 → 1845 rows
  4250 train: 2677 → 2677 rows
  4250 test: 1824 → 1824 rows
  2280 train: 3189 → 3189 rows
  2280 test: 1846 → 1846 rows
  4020 train: 4141 → 4141 rows
  4020 test: 1841 → 1841 rows
  3080 train: 3601 → 3601 rows
  3080 test: 1828 → 1828 rows
  8160 train: 2605 → 2605 rows
  8160 test: 1832 → 1832 rows

═══════════════════════════════════════════════════════
  Optuna search — 2060
═══════════════════════════════════════════════════════


[I 2026-06-15 09:00:57,754] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.8, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-15 09:01:13,162] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 1.4, 'gamma': 0.5}. Best is trial 0 with value: -1.0.
[I 2026-06-15 09:01:46,170] Trial 2 finished with value: -0.27040286715731626 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 1.9000000000000001, 'gamma': 1.0}. Best is trial 2 with value: -0.27040286715731626.
[I 2026-06-15 09:02:01,332


  Best joint score : 0.1914
  Best trial       : 35
  Best params:
    hidden_size     : 64
    num_layers      : 3
    dropout         : 0.30000000000000004
    lr              : 0.0027600111223192413
    batch_size      : 32
    seq_len         : 30
    alpha           : 5.0
    beta            : 0.8
    move_threshold  : 1.7000000000000002
    gamma           : 2.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
35      35  0.191414                  64                  3             0.3   
27      27  0.130980                  64                  2             0.3   
44      44  0.111725                  64                  2             0.3   
41      41 -0.191943                  64                  2             0.3   
31      31 -0.209484                  64                  2             0.3   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
35   0.002760                 32              30         

...


  Best joint score : 0.0877
  Best trial       : 2
  Best params:
    hidden_size     : 64
    num_layers      : 2
    dropout         : 0.4
    lr              : 0.0043709904681305065
    batch_size      : 32
    seq_len         : 20
    alpha           : 7.5
    beta            : 0.3
    move_threshold  : 1.9000000000000001
    gamma           : 1.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
2        2  0.087715                  64                  2             0.4   
47      47  0.083199                  64                  2             0.4   
23      23  0.080784                  64                  2             0.2   
43      43  0.071970                  64                  2             0.4   
17      17  0.063801                  64                  2             0.4   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
2    0.004371                 32              20           7.5          0.

[I 2026-06-15 10:37:05,874] Trial 0 finished with value: -0.6666666666666666 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.8, 'gamma': 1.0}. Best is trial 0 with value: -0.6666666666666666.
[I 2026-06-15 10:37:22,245] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 1.4, 'gamma': 0.5}. Best is trial 0 with value: -0.6666666666666666.
[I 2026-06-15 10:37:42,275] Trial 2 finished with value: -0.6666666666666666 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 1.9000000000000001, 'gamma': 1.0}. Best is trial 0 with value: -0.6


  Best joint score : 0.0700
  Best trial       : 46
  Best params:
    hidden_size     : 64
    num_layers      : 2
    dropout         : 0.30000000000000004
    lr              : 0.0017421738251537956
    batch_size      : 32
    seq_len         : 20
    alpha           : 5.5
    beta            : 0.4
    move_threshold  : 1.5
    gamma           : 3.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
46      46  0.070002                  64                  2             0.3   
22      22  0.051155                  64                  2             0.2   
27      27  0.025264                  64                  2             0.2   
13      13  0.014798                  64                  2             0.4   
34      34  0.008815                  64                  2             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
46   0.001742                 32              20           5.5          

[I 2026-06-15 10:56:31,240] Trial 0 finished with value: 0.18397237354747586 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.8, 'gamma': 1.0}. Best is trial 0 with value: 0.18397237354747586.
[I 2026-06-15 10:57:16,024] Trial 1 finished with value: 0.08449520876327692 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 1.4, 'gamma': 0.5}. Best is trial 0 with value: 0.18397237354747586.
[I 2026-06-15 10:57:52,819] Trial 2 finished with value: -0.21723817308450152 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 1.9000000000000001, 'gamma': 1.0}. Best is trial 0 


  Best joint score : 0.2783
  Best trial       : 44
  Best params:
    hidden_size     : 64
    num_layers      : 3
    dropout         : 0.2
    lr              : 0.0007689595445172317
    batch_size      : 32
    seq_len         : 10
    alpha           : 5.5
    beta            : 0.3
    move_threshold  : 0.9
    gamma           : 3.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
44      44  0.278310                  64                  3             0.2   
45      45  0.264378                  64                  3             0.2   
47      47  0.260036                  64                  3             0.2   
49      49  0.259887                  64                  3             0.3   
13      13  0.259286                  64                  3             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
44   0.000769                 32              10           5.5          0.3   
45   0.00

In [7]:
# Training 
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import os
import time

trained_models = {}

def compute_joint_score(price_dir, price_corr,
                          pred_up_pct, std_ratio):
      bias_gap        = abs(pred_up_pct - 0.5)
      std_penalty     = abs(1.0 - std_ratio) * 0.3
      balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0
      return (
          0.40 * price_dir +
          0.25 * max(price_corr,  0.0) +
          std_penalty - bias_gap * 3.0 - balance_penalty
      )


def compute_val_metrics(model, loader, device):
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in loader:
            all_preds.append(model(X_batch.to(device)).cpu().numpy())
            all_actuals.append(y_batch.numpy())

    preds   = np.vstack(all_preds)
    actuals = np.vstack(all_actuals)

    price_dir  = balanced_accuracy_score(
        np.sign(actuals[:, 0]).astype(int),
        np.sign(preds[:, 0]).astype(int)
    )

    def safe_corr(a, b):
        if np.std(a) < 1e-8 or np.std(b) < 1e-8:
            return 0.0
        r = np.corrcoef(a, b)[0, 1]
        return 0.0 if np.isnan(r) else float(r)

    price_corr  = safe_corr(actuals[:, 0], preds[:, 0])

    pred_up_pct  = (preds[:, 0] > 0).mean()
    std_ratio    = np.std(preds[:, 0]) / max(np.std(actuals[:, 0]), 1e-6)
    std_penalty  = abs(1.0 - std_ratio) * 0.2
    collapse_penalty = 1.0 if (pred_up_pct < 0.05 or pred_up_pct > 0.95) else 0.0

    joint_score = (
        0.35 * price_dir               +
        0.25 * max(price_corr,  0.0)   -
        std_penalty                    -
        collapse_penalty
    )

    return joint_score, price_dir, price_corr, pred_up_pct, std_ratio


# ── Training loop ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*72}")
    print(f"  Training — {symbol}")
    print(f"{'═'*72}")

    best = best_params_all[symbol]
    print(f"  Best params: {best}")

    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild sequences with best seq_len ───────────────────────────────────
    X_train_f, y_train_f, w_train_f = create_sequences(data["train_X_scaled"], data["train_y_scaled"], best["seq_len"], move_threshold)
    X_val_f,   y_val_f,   _         = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)
    X_test_f,  y_test_f,  _         = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)

    # ── Loaders ───────────────────────────────────────────────────────────────
    train_loader = DataLoader(
        TadawulDataset(X_train_f, y_train_f, w_train_f),
        batch_size=best["batch_size"],
        shuffle=True,
        drop_last=True
    )
    val_loader  = DataLoader(TadawulDataset(X_val_f,  y_val_f),  batch_size=best["batch_size"], shuffle=False)
    test_loader = DataLoader(TadawulDataset(X_test_f, y_test_f), batch_size=best["batch_size"], shuffle=False)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"]
    ).to(device)

    WARMUP_EPOCHS    = 10
    BASE_ALPHA       = best["alpha"]
    BASE_BETA        = best["beta"]
    BASE_GAMMA       = best.get("gamma", 1.0)
    optimizer        = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
    scheduler        = ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

    EPOCHS           = 100
    PATIENCE         = 15
    best_joint_score = -np.inf
    patience_ctr     = 0
    history          = {"train_loss": [], "val_loss": [], "joint_score": [], "price_dir": [], "price_corr": []}

    os.makedirs(f"models/{symbol}", exist_ok=True)
    save_path = f"models/{symbol}/{symbol}_best_bilstm.pt"

    print(f"\n{'Epoch':>6} | {'TrLoss':>8} | {'VaLoss':>8} | "
          f"{'Joint':>7} | {'PDir':>6} | "
          f"{'Pr':>6} | {'UP%':>5} | {'StdR':>5} | {'s':>5}")
    print("─" * 90)
    
    training_state = type('State', (), {})() 
    
    for epoch in range(1, EPOCHS + 1):
        start        = time.time()
        warmup_scale = 0.3 + 0.7 * min(1.0, epoch / WARMUP_EPOCHS) 
        criterion    = JointPredictionLoss(
            alpha = BASE_ALPHA * warmup_scale,
            beta  = BASE_BETA,
            gamma = BASE_GAMMA
        )

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for X_batch, y_batch, w_batch in train_loader:
            X_batch, y_batch, w_batch = X_batch.to(device), y_batch.to(device), w_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # ── Val loss ──────────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch, _ in val_loader:
                val_losses.append(
                    criterion(model(X_batch.to(device)), y_batch.to(device)).item()
                )

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)

        joint_score, p_dir, p_corr, pred_up_pct, std_ratio = \
            compute_val_metrics(model, val_loader, device)

        elapsed = time.time() - start
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["joint_score"].append(joint_score)
        history["price_dir"].append(p_dir)
        history["price_corr"].append(p_corr)

        print(f"{epoch:>6} | {train_loss:>8.5f} | {val_loss:>8.5f} | "f"{joint_score:>7.4f} | {p_dir:>6.3f} | "f"{p_corr:>6.3f} | {pred_up_pct:>4.0%} | "f"{std_ratio:>5.2f} | {elapsed:>4.1f}s")

        # ── Collapse detection with restart ───────────────────────────────────
        if epoch > WARMUP_EPOCHS:
            if pred_up_pct > 0.85 or pred_up_pct < 0.15:
                consecutive_collapse = getattr(training_state, 'collapse_ctr', 0) + 1
                training_state.collapse_ctr = consecutive_collapse
                if consecutive_collapse >= 5:
                    print(f"\n  Collapse detected at epoch {epoch} "
                            f"(up%={pred_up_pct:.0%}, std_r={std_ratio:.2f}) — resetting weights")
                    model.apply(lambda m: m.reset_parameters()
                                if hasattr(m, 'reset_parameters') else None)
                    for pg in optimizer.param_groups:
                        pg['lr'] = best["lr"]
                    training_state.collapse_ctr = 0
            else:
                training_state.collapse_ctr = 0

        scheduler.step(joint_score)

        if joint_score > best_joint_score:
            best_joint_score = joint_score
            patience_ctr     = 0
            torch.save(model.state_dict(), save_path)
            print(f"         ✓ Saved  "
                  f"(p_dir={p_dir:.3f}, "
                  f"p_corr={p_corr:.3f}, up%={pred_up_pct:.0%}, std_r={std_ratio:.2f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    trained_models[symbol] = {
        "best_joint_score": best_joint_score,
        "history":          history,
    }
    print(f"\n  Best joint score: {best_joint_score:.4f}")

print(f"\n── Training done for {len(trained_models)} symbols ──")
for s, r in trained_models.items():
    print(f"  {s}: best joint score = {r['best_joint_score']:.4f}")


════════════════════════════════════════════════════════════════════════
  Training — 2060
════════════════════════════════════════════════════════════════════════
  Best params: {'hidden_size': 64, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0027600111223192413, 'batch_size': 32, 'seq_len': 30, 'alpha': 5.0, 'beta': 0.8, 'move_threshold': 1.7000000000000002, 'gamma': 2.0}

 Epoch |   TrLoss |   VaLoss |   Joint |   PDir |     Pr |   UP% |  StdR |     s
──────────────────────────────────────────────────────────────────────────────────────────
     1 |  0.42528 |  0.26515 | -0.9887 |  0.500 |  0.112 |   0% |  0.04 |  1.3s
         ✓ Saved  (p_dir=0.500, p_corr=0.112, up%=0%, std_r=0.04)
     2 |  0.40661 |  0.26927 | -0.0140 |  0.491 | -0.041 |   7% |  0.07 |  1.4s
         ✓ Saved  (p_dir=0.491, p_corr=-0.041, up%=7%, std_r=0.07)
     3 |  0.39313 |  0.27204 |  0.0325 |  0.520 |  0.064 |  10% |  0.17 |  1.3s
         ✓ Saved  (p_dir=0.520, p_corr=0.064, up%=10%, std_r=0.1

DIAGNOSTIC

In [8]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

def evaluate(actuals, preds, label):
    mae      = np.mean(np.abs(actuals - preds))
    rmse     = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc  = np.mean(np.sign(actuals) == np.sign(preds)) * 100

    corr       = np.corrcoef(actuals, preds)[0, 1]
    pred_std   = np.std(preds)
    actual_std = np.std(actuals)
    within_1pct = np.mean(np.abs(actuals - preds) < 1.0) * 100
    within_2pct = np.mean(np.abs(actuals - preds) < 2.0) * 100

    print(f"\n  [{label}]")
    print(f"  MAE             : {mae:.4f}%")
    print(f"  RMSE            : {rmse:.4f}%")
    print(f"  Pearson r       : {corr:.4f}")
    print(f"  Pred std        : {pred_std:.4f}  Actual std: {actual_std:.4f}")
    print(f"  Within ±1%      : {within_1pct:.1f}%")
    print(f"  Within ±2%      : {within_2pct:.1f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")

for symbol in symbols:
    if symbol not in prepared:
        print(f"\n  Skipping {symbol} — not in prepared dict.")
        continue
    if symbol not in trained_models:
        print(f"\n  Skipping {symbol} — no trained model found.")
        continue

    print(f"\n{'═'*55}")
    print(f"  Evaluating — {symbol}")
    print(f"{'═'*55}")

    best = best_params_all[symbol]
    data = prepared[symbol]
    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild test loader with best seq_len ─────────────────────────────────
    X_test_f, y_test_f, _ = create_sequences(          # ← unpack 3, discard w
        data["test_X_scaled"], data["test_y_scaled"], best["seq_len"], move_threshold
    )
    test_loader = DataLoader(
        TadawulDataset(X_test_f, y_test_f),            # no weights needed for eval
        batch_size=best["batch_size"],
        shuffle=False
    )

    # ── Load best model ───────────────────────────────────────────────────────
    checkpoint = torch.load(
        f"models/{symbol}/{symbol}_best_bilstm.pt",
        weights_only=True
    )

    inferred_hidden_size = checkpoint["lstm.weight_ih_l0"].shape[0] // 4
    inferred_num_layers  = sum(
        1 for k in checkpoint if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = inferred_hidden_size,
        num_layers  = inferred_num_layers,
        dropout     = best["dropout"]
    ).to(device)

    model.load_state_dict(checkpoint)
    model.eval()

    # ── Load target scaler ────────────────────────────────────────────────────
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        target_scaler = pickle.load(f)

    # ── Inference ─────────────────────────────────────────────────────────────
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:        # ← unpack 3
            preds   = model(X_batch.to(device)).cpu().numpy()
            actuals = y_batch.numpy()
            all_preds.append(preds)
            all_actuals.append(actuals)

    all_preds   = np.vstack(all_preds)
    all_actuals = np.vstack(all_actuals)

    # ── Inverse transform ─────────────────────────────────────────────────────
    all_preds   = target_scaler.inverse_transform(all_preds)
    all_actuals = target_scaler.inverse_transform(all_actuals)

    price_preds_pct    = all_preds[:, 0]
    price_actuals_pct  = all_actuals[:, 0]

    # ── Metrics ───────────────────────────────────────────────────────────
    evaluate(price_actuals_pct,  price_preds_pct,  "Price 5d % Change")

    # ── Save predictions ─────────────────────────────────────────────────────
    results_df = pd.DataFrame({
        "actual_price_pct"    : price_actuals_pct,
        "predicted_price_pct" : price_preds_pct,
    })
    out_path = f"models/{symbol}/{symbol}_test_predictions.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\n  Predictions saved: {out_path}")

print(f"\n── Evaluation done for {len(symbols)} symbols ──")


EVALUATION ON TEST SET

═══════════════════════════════════════════════════════
  Evaluating — 2060
═══════════════════════════════════════════════════════

  [Price 5d % Change]
  MAE             : 4.2932%
  RMSE            : 5.4632%
  Pearson r       : -0.0012
  Pred std        : 3.3970  Actual std: 4.0677
  Within ±1%      : 15.2%
  Within ±2%      : 29.6%
  Directional Acc : 52.40%

  Predictions saved: models/2060/2060_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — 4250
═══════════════════════════════════════════════════════

  [Price 5d % Change]
  MAE             : 4.4248%
  RMSE            : 5.5306%
  Pearson r       : 0.0128
  Pred std        : 3.3435  Actual std: 4.2617
  Within ±1%      : 13.4%
  Within ±2%      : 27.0%
  Directional Acc : 48.34%

  Predictions saved: models/4250/4250_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — 2280
════════════════════════════════════════════════

In [9]:
import pandas as pd
import numpy as np

for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0

    print(f"\n{'═'*55}")
    print(f"  {symbol} — Honest Model Baseline")
    print(f"{'═'*55}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size:")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 100.0]
    labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
    price_actual_abs = price_actual.abs()
    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean() * 100
            print(f"    {label:>8} moves : {acc:.2f}%  ({mask.sum()} samples)")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")


═══════════════════════════════════════════════════════
  2060 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0208   std: 4.0688
  Predicted mean : -1.3352   std: 3.3979

  Overall Dir Acc : 52.40%
  When actual UP  : 37.40%  (869 samples)
  When actual DOWN: 66.17%  (946 samples)

  % predicted UP  : 35.54%
  % actual UP     : 47.88%

  Dir Acc by Move Size:
       <0.5% moves : 54.40%  (182 samples)
      0.5-1% moves : 58.05%  (205 samples)
        1-2% moves : 53.41%  (352 samples)
        2-5% moves : 52.91%  (705 samples)
         >5% moves : 46.36%  (371 samples)

═══════════════════════════════════════════════════════
  4250 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.2296   std: 4.2629
  Predicted mean : 1.0400   std: 3.3444

  Overall Dir Acc : 48.34%
  When actual UP  : 60.35%  (797 samples)
  When actual DOWN: 38.83%  (1007 samples)

  % predicted UP  : 60.81%
 